# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")


✅ Libraries imported successfully!
Boto3 version: 1.37.3
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [3]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [4]:

EVENT_NAME = '202406_Flood_IA'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'sentinel2'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [5]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [6]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

✅ S3 client initialized successfully
   Found 68 accessible buckets
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 49 .tif files in the S3 bucket.


['drcs_activations/202406_Flood_IA/sentinel2/T14TPN_20240610T171859_TCI_10m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TPN_20240610T17859_SWI_20m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TPN_20240625T172001_TCI_10m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TPP_20240625T172001_TCI_10m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TQN_20240523T170851_SWI.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TQN_20240523T170851_SWI_2.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TQN_20240528T170849_SWI_20m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TQN_20240610T17859_SWI_20m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TQN_20240625T172001_TCI_10m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TQP_20240610T17859_SWI_20m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T15TUH_20240528T170849_SWI_20m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T15TUJ_20240523T170851_SWI_.tif',
 'drcs_activations/202406_Fl

## Configure bucket and paths (no need to create session manually)

In [7]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

## Define Chunked COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with:
- Chunked processing to handle large files
- Memory monitoring
- Progress tracking
- Proper CRS and caching

In [8]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 121
  - Total size: 6.65 GB

📁 Cached files (first 10):
  - drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023126133547_aid0001.tif (1.6 MB)
  - drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023127043620_aid0001.tif (0.5 MB)
  - drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023127043712_aid0001.tif (12.1 MB)
  - drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023131025921_aid0001.tif (0.4 MB)
  - drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023131030013_aid0001.tif (28.9 MB)
  - drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023134021109_aid0001.tif (0.4 MB)
  - drcs_activations/202405_Heat_TX/ECOSTRESS

(121, 7135407245)

In [9]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    _
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [10]:
keys


['drcs_activations/202406_Flood_IA/sentinel2/T14TPN_20240610T171859_TCI_10m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TPN_20240610T17859_SWI_20m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TPN_20240625T172001_TCI_10m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TPP_20240625T172001_TCI_10m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TQN_20240523T170851_SWI.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TQN_20240523T170851_SWI_2.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TQN_20240528T170849_SWI_20m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TQN_20240610T17859_SWI_20m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TQN_20240625T172001_TCI_10m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TQP_20240610T17859_SWI_20m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T15TUH_20240528T170849_SWI_20m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T15TUJ_20240523T170851_SWI_.tif',
 'drcs_activations/202406_Fl

In [11]:
def create_cog_filename_sentinel2_datetime(f, EVENT_NAME):
    """Create COG filename for Sentinel-2 files with datetime at the end."""
    from pathlib import Path
    import re
    
    filename = Path(f).stem
    extension = Path(f).suffix
    
    # Parse Sentinel-2 filename patterns:
    # T15TXH_20240626T170051_TCI_10m
    # T14TPN_20240610T17859_SWI_20m (missing minutes/seconds)
    # T15TXF_202400527T170051_SWI_20m (typo in date)
    
    parts = filename.split('_')
    
    if len(parts) >= 3:
        tile = parts[0]  # T15TXH
        datetime_str = parts[1]  # 20240626T170051
        
        # Handle the datetime string
        if 'T' in datetime_str:
            date_part, time_part = datetime_str.split('T')
            
            # Fix common issues in date
            # Handle case like "202400527" (should be "20240527")
            if len(date_part) == 9 and date_part[4:6] == '00':
                date_part = date_part[:4] + date_part[6:]
            
            # Format date
            if len(date_part) == 8:
                formatted_date = f"{date_part[:4]}-{date_part[4:6]}-{date_part[6:8]}"
            else:
                formatted_date = date_part
            
            # Handle time part
            if len(time_part) == 6:  # HHMMSS
                formatted_time = f"{time_part[:2]}:{time_part[2:4]}:{time_part[4:6]}"
            elif len(time_part) == 5:  # HHMSS (missing leading zero)
                formatted_time = f"{time_part[:2]}:{time_part[2:4]}:0{time_part[4]}"
            elif len(time_part) == 4:  # HHMM
                formatted_time = f"{time_part[:2]}:{time_part[2:4]}:00"
            else:
                formatted_time = time_part
            
            formatted_datetime = f"{formatted_date}T{formatted_time}Z"
        else:
            # No time component
            if len(datetime_str) == 8:
                formatted_datetime = f"{datetime_str[:4]}-{datetime_str[4:6]}-{datetime_str[6:8]}T00:00:00Z"
            else:
                formatted_datetime = datetime_str
        
        # Collect remaining parts
        remaining_parts = parts[2:]
        
        # Build new filename
        new_parts = [EVENT_NAME, tile]
        new_parts.extend(remaining_parts)
        new_parts.append(formatted_datetime)
        
        cog_filename = '_'.join(new_parts) + extension
    else:
        # Fallback
        cog_filename = f'{EVENT_NAME}_{filename}{extension}'
    
    return cog_filename

filter_str = 'TCI'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_sentinel2_datetime(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")



Testing WM filename:
  202406_Flood_IA_T14TPN_TCI_10m_2024-06-10T17:18:59Z.tif
  202406_Flood_IA_T14TPN_TCI_10m_2024-06-25T17:20:01Z.tif
  202406_Flood_IA_T14TPP_TCI_10m_2024-06-25T17:20:01Z.tif
  202406_Flood_IA_T14TQN_TCI_10m_2024-06-25T17:20:01Z.tif
  202406_Flood_IA_T15TUJ_TCI_2024-05-23T17:08:51Z.tif
  202406_Flood_IA_T15TVJ_TCI_10m_2024-06-14T16:58:49Z.tif
  202406_Flood_IA_T15TVJ_TCI_10m_2024-06-24T16:58:49Z.tif
  202406_Flood_IA_T15TWF_TCI_10m_2024-05-30T16:58:51Z.tif
  202406_Flood_IA_T15TWF_TCI_10m_2024-06-24T16:58:49Z.tif
  202406_Flood_IA_T15TWG_TCI_10m_2024-05-30T16:58:51Z.tif
  202406_Flood_IA_T15TWG_TCI_10m_2024-06-24T16:58:49Z.tif
  202406_Flood_IA_T15TWH_TCI_10m_2024-05-30T16:58:51Z.tif
  202406_Flood_IA_T15TWH_TCI_10m_2024-06-24T16:58:49Z.tif
  202406_Flood_IA_T15TWJ_TCI_10m_2024-06-14T16:58:49Z.tif
  202406_Flood_IA_T15TWJ_TCI_10m_2024-06-24T16:58:49Z.tif
  202406_Flood_IA_T15TWK_TCI_10m_2024-06-14T16:58:49Z.tif
  202406_Flood_IA_T15TWK_TCI_10m_2024-06-24T16:58:49Z.t

In [12]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_sentinel2_datetime, 
                                target_dir = "Sentinel-2/TCI", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202406_Flood_IA_T14TPN_TCI_10m_2024-06-10T17:18:59Z.tif
  202406_Flood_IA_T14TPN_TCI_10m_2024-06-25T17:20:01Z.tif
  202406_Flood_IA_T14TPP_TCI_10m_2024-06-25T17:20:01Z.tif
  202406_Flood_IA_T14TQN_TCI_10m_2024-06-25T17:20:01Z.tif
  202406_Flood_IA_T15TUJ_TCI_2024-05-23T17:08:51Z.tif
  202406_Flood_IA_T15TVJ_TCI_10m_2024-06-14T16:58:49Z.tif
  202406_Flood_IA_T15TVJ_TCI_10m_2024-06-24T16:58:49Z.tif
  202406_Flood_IA_T15TWF_TCI_10m_2024-05-30T16:58:51Z.tif
  202406_Flood_IA_T15TWF_TCI_10m_2024-06-24T16:58:49Z.tif
  202406_Flood_IA_T15TWG_TCI_10m_2024-05-30T16:58:51Z.tif
  202406_Flood_IA_T15TWG_TCI_10m_2024-06-24T16:58:49Z.tif
  202406_Flood_IA_T15TWH_TCI_10m_2024-05-30T16:58:51Z.tif
  202406_Flood_IA_T15TWH_TCI_10m_2024-06-24T16:58:49Z.tif
  202406_Flood_IA_T15TWJ_TCI_10m_2024-06-14T16:58:49Z.tif
  202406_Flood_IA_T15TWJ_TCI_10m_2024-06-24T16:58:49Z.tif
  202406_Flood_IA_T15TWK_TCI_10m_2024-06-14T16:58:49Z.tif
  202406_Flood_IA_T15TWK_TCI_10m_2024-06-24T16:58:49Z.tif

Band 1:  61%|██████    | 79/130 [00:03<00:01, 26.04chunks/s]


   [MEMORY] High usage: 590.4 MB, forcing cleanup...


Band 1:  67%|██████▋   | 87/130 [00:03<00:01, 23.59chunks/s]


   [MEMORY] High usage: 630.4 MB, forcing cleanup...


...

   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=10, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=19, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=12, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpd8daknpp_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp2orb1mgo.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-2/TCI/202406_Flood_IA_T15XG_TCI_10m_2024-06-26T17:00:51Z.tif
   [MEMORY] Final: 1517.9 MB (Change: -0.1 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202406_Flood_IA_T15XG_TCI_10m_2024-06-26T17:00:51Z.tif

✅ Batch processing complete: 23 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Sentinel-2/TCI/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Sentinel-2/TCI/files_converted.csv
📁 COGs saved locally to: output/202406_Flood_IA

📊 BATCH PROCESSING SUMMARY
Total files processed: 23
Successful: 23
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-09T23:19:43.817967


In [13]:
keys


['drcs_activations/202406_Flood_IA/sentinel2/T14TPN_20240610T171859_TCI_10m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TPN_20240610T17859_SWI_20m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TPN_20240625T172001_TCI_10m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TPP_20240625T172001_TCI_10m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TQN_20240523T170851_SWI.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TQN_20240523T170851_SWI_2.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TQN_20240528T170849_SWI_20m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TQN_20240610T17859_SWI_20m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TQN_20240625T172001_TCI_10m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T14TQP_20240610T17859_SWI_20m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T15TUH_20240528T170849_SWI_20m.tif',
 'drcs_activations/202406_Flood_IA/sentinel2/T15TUJ_20240523T170851_SWI_.tif',
 'drcs_activations/202406_Fl

In [14]:
filter_str = 'SWI'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_sentinel2_datetime(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")


Testing WM filename:
  202406_Flood_IA_T14TPN_SWI_20m_2024-06-10T17:85:09Z.tif
  202406_Flood_IA_T14TQN_SWI_2024-05-23T17:08:51Z.tif
  202406_Flood_IA_T14TQN_SWI_2_2024-05-23T17:08:51Z.tif
  202406_Flood_IA_T14TQN_SWI_20m_2024-05-28T17:08:49Z.tif
  202406_Flood_IA_T14TQN_SWI_20m_2024-06-10T17:85:09Z.tif
  202406_Flood_IA_T14TQP_SWI_20m_2024-06-10T17:85:09Z.tif
  202406_Flood_IA_T15TUH_SWI_20m_2024-05-28T17:08:49Z.tif
  202406_Flood_IA_T15TUJ_SWI__2024-05-23T17:08:51Z.tif
  202406_Flood_IA_T15TVJ_SWI_20m_2024-06-14T16:58:49Z.tif
  202406_Flood_IA_T15TVJ_SWI_20m_2024-06-24T16:58:49Z.tif
  202406_Flood_IA_T15TWF_SWI_20m_2024-05-30T16:58:51Z.tif
  202406_Flood_IA_T15TWF_SWI_20m_2024-06-24T16:58:49Z.tif
  202406_Flood_IA_T15TWG_SWI_20m_2024-05-30T16:58:51Z.tif
  202406_Flood_IA_T15TWG_SWI_20m_2024-06-24T16:58:49Z.tif
  202406_Flood_IA_T15TWH_SWI_20m_2024-05-30T16:58:51Z.tif
  202406_Flood_IA_T15TWH_SWI_20m_2024-06-24T16:58:49Z.tif
  202406_Flood_IA_T15TWJ_SWI_20m_2024-06-14T16:58:49Z.tif
  

In [15]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_sentinel2_datetime, 
                                target_dir = "Sentinel-2/SWI", 
                                EVENT_NAME = EVENT_NAME)

Testing filenames:
  202406_Flood_IA_T14TPN_SWI_20m_2024-06-10T17:85:09Z.tif
  202406_Flood_IA_T14TQN_SWI_2024-05-23T17:08:51Z.tif
  202406_Flood_IA_T14TQN_SWI_2_2024-05-23T17:08:51Z.tif
  202406_Flood_IA_T14TQN_SWI_20m_2024-05-28T17:08:49Z.tif
  202406_Flood_IA_T14TQN_SWI_20m_2024-06-10T17:85:09Z.tif
  202406_Flood_IA_T14TQP_SWI_20m_2024-06-10T17:85:09Z.tif
  202406_Flood_IA_T15TUH_SWI_20m_2024-05-28T17:08:49Z.tif
  202406_Flood_IA_T15TUJ_SWI__2024-05-23T17:08:51Z.tif
  202406_Flood_IA_T15TVJ_SWI_20m_2024-06-14T16:58:49Z.tif
  202406_Flood_IA_T15TVJ_SWI_20m_2024-06-24T16:58:49Z.tif
  202406_Flood_IA_T15TWF_SWI_20m_2024-05-30T16:58:51Z.tif
  202406_Flood_IA_T15TWF_SWI_20m_2024-06-24T16:58:49Z.tif
  202406_Flood_IA_T15TWG_SWI_20m_2024-05-30T16:58:51Z.tif
  202406_Flood_IA_T15TWG_SWI_20m_2024-06-24T16:58:49Z.tif
  202406_Flood_IA_T15TWH_SWI_20m_2024-05-30T16:58:51Z.tif
  202406_Flood_IA_T15TWH_SWI_20m_2024-06-24T16:58:49Z.tif
  202406_Flood_IA_T15TWJ_SWI_20m_2024-06-14T16:58:49Z.tif
  20

   [BAND 2/3] Processing...


   [BAND 3/3] Processing...


   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=1082, max=9489, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=920, max=7735, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=1191, max=7464, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint16
   [NODATA] Using nodata value 0 for uint16 data
   [PREDICTOR] Data type: uint16, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmp2v3k78u7_temp.tif



...

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=952, max=8678, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=925, max=8850, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=1029, max=8458, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint16
   [NODATA] Using nodata value 0 for uint16 data
   [PREDICTOR] Data type: uint16, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmp3eeel3o1_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp76drltki.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-2/SWI/202406_Flood_IA_T15XG_SWI_20m_2024-06-26T17:00:51Z.tif
   [MEMORY] Final: 1557.9 MB (Change: +0.3 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202406_Flood_IA_T15XG_SWI_20m_2024-06-26T17:00:51Z.tif

✅ Batch processing complete: 26 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Sentinel-2/SWI/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Sentinel-2/SWI/files_converted.csv
📁 COGs saved locally to: output/202406_Flood_IA

📊 BATCH PROCESSING SUMMARY
Total files processed: 26
Successful: 26
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-09T23:32:13.320165


## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [16]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")


📊 Memory Usage Summary:
  Current memory usage: 1557.9 MB
  Available memory: 24509.1 MB
  Memory percent used: 22.5%
